In [ ]:
# CELL 1: Install libraries
!pip install pandas numpy matplotlib seaborn scikit-learn joblib

print("✅ Done! Libraries ready.")

In [ ]:
# CELL 2: Import all tools
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import warnings
warnings.filterwarnings('ignore')

print("✅ All imports successful!")

In [ ]:
# CELL 3: Load diabetes dataset
df = pd.read_csv('diabetes_prediction_dataset.csv')

print("✅ Dataset loaded!")
print(f"Rows: {df.shape[0]},  Columns: {df.shape[1]}")
print("\nFirst 5 rows:")
df.head()

In [ ]:
# CELL 4: Understand the data

print("=== COLUMN NAMES & DATA TYPES ===")
print(df.dtypes)

print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== TARGET COLUMN (diabetes) ===")
print(df['diabetes'].value_counts())
print("\n0 = No Diabetes,  1 = Has Diabetes")

print("\n=== BASIC STATISTICS ===")
df.describe()

In [ ]:
# CELL 5: Data visualization

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Chart 1: Target distribution
df['diabetes'].value_counts().plot(kind='bar', ax=axes[0],
    color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Diabetes Distribution\n(0=No, 1=Yes)', fontsize=13)
axes[0].set_xlabel('Diabetes')
axes[0].set_ylabel('Count')
axes[0].tick_params(rotation=0)

# Chart 2: Age distribution
df['age'].plot(kind='hist', ax=axes[1], bins=30,
    color='steelblue', edgecolor='black')
axes[1].set_title('Age Distribution of Patients', fontsize=13)
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Count')

# Chart 3: BMI vs Blood Glucose (colored by diabetes)
scatter = axes[2].scatter(df['bmi'], df['blood_glucose_level'],
    c=df['diabetes'], cmap='coolwarm', alpha=0.3, s=5)
axes[2].set_title('BMI vs Blood Glucose Level', fontsize=13)
axes[2].set_xlabel('BMI')
axes[2].set_ylabel('Blood Glucose Level')
plt.colorbar(scatter, ax=axes[2], label='Diabetes')

plt.tight_layout()
plt.savefig('diabetes_exploration.png', dpi=150)
plt.show()
print("✅ Charts saved!")

In [ ]:
# CELL 6: Data cleaning and encoding

# Step 6a: Encode 'gender' column (Male/Female → numbers)
le_gender = LabelEncoder()
df['gender'] = le_gender.fit_transform(df['gender'])
print("Gender values after encoding:", df['gender'].unique())

# Step 6b: Encode 'smoking_history' column (text → numbers)
le_smoke = LabelEncoder()
df['smoking_history'] = le_smoke.fit_transform(df['smoking_history'])
print("Smoking history values after encoding:", df['smoking_history'].unique())

print("\n✅ All text columns converted to numbers!")
print("\nFinal column types:")
print(df.dtypes)

In [ ]:
# CELL 7: Separate features (X) and target (y)

# X = everything the model learns FROM (inputs)
X = df.drop('diabetes', axis=1)

# y = what the model tries to PREDICT (output)
y = df['diabetes']

print("Feature columns (X):", X.columns.tolist())
print("\nTarget column (y): diabetes")
print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")

# Scale features (bring all values to same range)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\n✅ Features scaled successfully!")

In [ ]:
# CELL 8: Split data into training set and testing set
# 80% for training, 20% for testing

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,       # 20% for testing
    random_state=42      # fixed seed = same split every time
)

print(f"Training samples:  {X_train.shape[0]}")
print(f"Testing samples:   {X_test.shape[0]}")
print("\n✅ Data split done!")
print("Think of it like: 80% of data goes into study material, 20% becomes the exam.")

In [ ]:
# CELL 9: Train Random Forest model
# This is the actual machine learning step!

print("⏳ Training model... please wait...")

model = RandomForestClassifier(
    n_estimators=100,   # 100 decision trees working together
    random_state=42
)

model.fit(X_train, y_train)

print("✅ Model trained successfully!")
print("\nWhat just happened: The model studied 80,000 patient records")
print("and learned patterns that distinguish diabetic from non-diabetic patients.")

In [ ]:
# CELL 10: Evaluate the model on unseen test data

y_pred = model.predict(X_test)

# Accuracy score
acc = accuracy_score(y_test, y_pred)
print(f"🎯 MODEL ACCURACY: {acc * 100:.2f}%")
print("\n" + "="*45)
print("DETAILED CLASSIFICATION REPORT:")
print("="*45)
print(classification_report(y_test, y_pred,
      target_names=['No Diabetes', 'Has Diabetes']))

In [ ]:
# CELL 11: Confusion Matrix visualization

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Diabetes', 'Has Diabetes'],
            yticklabels=['No Diabetes', 'Has Diabetes'])
plt.title('Confusion Matrix — Diabetes Model', fontsize=14)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.tight_layout()
plt.savefig('diabetes_confusion_matrix.png', dpi=150)
plt.show()

# Explain the matrix
tn, fp, fn, tp = cm.ravel()
print(f"\n📊 Results Breakdown:")
print(f"  ✅ Correctly predicted NO diabetes:  {tn}")
print(f"  ✅ Correctly predicted HAS diabetes: {tp}")
print(f"  ❌ Said diabetes but actually not:   {fp}  (False Alarm)")
print(f"  ❌ Missed actual diabetes cases:     {fn}  (Missed Cases)")

In [ ]:
# CELL 12: Which features matter most?

feature_names = df.drop('diabetes', axis=1).columns
importances = model.feature_importances_

feat_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(feat_df['Feature'], feat_df['Importance'],
         color='steelblue', edgecolor='black')
plt.title('Feature Importance — Diabetes Model\n(What factors matter most?)', fontsize=13)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('diabetes_feature_importance.png', dpi=150)
plt.show()
print("✅ Top 3 most important features:")
print(feat_df.tail(3)[['Feature','Importance']].iloc[::-1].to_string(index=False))

In [ ]:
# CELL 13: Save model and scaler to files
# We need these files later for the web app

joblib.dump(model, 'diabetes_model.pkl')
joblib.dump(scaler, 'diabetes_scaler.pkl')

print("✅ Model saved as:  diabetes_model.pkl")
print("✅ Scaler saved as: diabetes_scaler.pkl")
print("\n📌 IMPORTANT: Download both .pkl files to your computer!")
print("   Left sidebar → find the files → right-click → Download")

In [ ]:
# CELL 14: Predict for a new patient manually

sample_patient = np.array([[
    1,      # gender: Male
    45,     # age
    1,      # hypertension: Yes
    0,      # heart_disease: No
    0,      # smoking_history (encoded)
    28.5,   # bmi
    6.5,    # HbA1c_level
    140     # blood_glucose_level
]])

# Scale it the same way
sample_scaled = scaler.transform(sample_patient)

# Predict
prediction = model.predict(sample_scaled)
probability = model.predict_proba(sample_scaled)

print("="*40)
print("   PREDICTION RESULT")
print("="*40)
if prediction[0] == 1:
    print("⚠️  Result: HIGH RISK of Diabetes")
else:
    print("✅  Result: LOW RISK of Diabetes")

print(f"\nConfidence:")
print(f"  No Diabetes:  {probability[0][0]*100:.1f}%")
print(f"  Has Diabetes: {probability[0][1]*100:.1f}%")

In [ ]:
# CELL 15: Load Heart Disease dataset

df_heart = pd.read_csv('HeartDiseaseTrain-Test.csv')

print("✅ Dataset loaded!")
print(f"Rows: {df_heart.shape[0]},  Columns: {df_heart.shape[1]}")
print("\nColumn names:")
print(df_heart.columns.tolist())
print("\nFirst 5 rows:")
df_heart.head()

In [ ]:
# CELL 16: Explore Heart Disease data

print("=== DATA TYPES ===")
print(df_heart.dtypes)

print("\n=== MISSING VALUES ===")
print(df_heart.isnull().sum())

print("\n=== TARGET COLUMN (target) ===")
print(df_heart['target'].value_counts())
print("\n0 = No Heart Disease,  1 = Has Heart Disease")

print("\n=== UNIQUE VALUES IN TEXT COLUMNS ===")
text_cols = df_heart.select_dtypes(include='object').columns
for col in text_cols:
    print(f"\n{col}: {df_heart[col].unique()}")

In [ ]:
# CELL 17: Visualize Heart Disease data

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Chart 1: Target distribution
df_heart['target'].value_counts().plot(kind='bar', ax=axes[0],
    color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Heart Disease Distribution\n(0=No, 1=Yes)', fontsize=13)
axes[0].set_xlabel('Heart Disease')
axes[0].set_ylabel('Count')
axes[0].tick_params(rotation=0)

# Chart 2: Age distribution
df_heart['age'].plot(kind='hist', ax=axes[1], bins=25,
    color='tomato', edgecolor='black')
axes[1].set_title('Age Distribution of Patients', fontsize=13)
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Count')

# Chart 3: Age vs Max Heart Rate colored by disease
scatter = axes[2].scatter(
    df_heart['age'],
    df_heart['Max_heart_rate'],
    c=df_heart['target'],
    cmap='coolwarm', alpha=0.6, s=40, edgecolors='k', linewidths=0.3
)
axes[2].set_title('Age vs Max Heart Rate', fontsize=13)
axes[2].set_xlabel('Age')
axes[2].set_ylabel('Max Heart Rate')
plt.colorbar(scatter, ax=axes[2], label='Heart Disease')

plt.tight_layout()
plt.savefig('heart_exploration.png', dpi=150)
plt.show()
print("✅ Charts saved!")

In [ ]:
# CELL 18: Encode text columns for Heart Disease

# We store each encoder separately so we can decode later if needed
encoders_heart = {}

text_cols = [
    'sex',
    'chest_pain_type',
    'fasting_blood_sugar',
    'rest_ecg',
    'exercise_induced_angina',
    'slope',
    'vessels_colored_by_flourosopy',
    'thalassemia'
]

df_heart_clean = df_heart.copy()   # work on a copy to keep original safe

for col in text_cols:
    le = LabelEncoder()
    df_heart_clean[col] = le.fit_transform(df_heart_clean[col])
    encoders_heart[col] = le   # save encoder for reference
    print(f"✅ {col}: {le.classes_}  →  {list(range(len(le.classes_)))}")

print("\n✅ All text columns encoded!")
print("\nFinal column types:")
print(df_heart_clean.dtypes)

In [ ]:
# CELL 19: Separate X (features) and y (target)

X_heart = df_heart_clean.drop('target', axis=1)
y_heart = df_heart_clean['target']

print("Feature columns:")
for i, col in enumerate(X_heart.columns):
    print(f"  {i+1}. {col}")

print(f"\nTotal features: {X_heart.shape[1]}")
print(f"Total samples:  {X_heart.shape[0]}")

# Scale features
scaler_heart = StandardScaler()
X_heart_scaled = scaler_heart.fit_transform(X_heart)

print("\n✅ Features prepared and scaled!")

In [ ]:
# CELL 20: Train/Test split

X_heart_train, X_heart_test, y_heart_train, y_heart_test = train_test_split(
    X_heart_scaled, y_heart,
    test_size=0.2,
    random_state=42
)

print(f"Training samples:  {X_heart_train.shape[0]}")
print(f"Testing samples:   {X_heart_test.shape[0]}")
print("\n✅ Split done! (80% train / 20% test)")

In [ ]:
# CELL 21: Train Random Forest model for Heart Disease

print("⏳ Training Heart Disease model...")

model_heart = RandomForestClassifier(
    n_estimators=200,    # more trees = better accuracy for smaller dataset
    max_depth=10,        # prevents overfitting
    random_state=42
)

model_heart.fit(X_heart_train, y_heart_train)

print("✅ Model trained successfully!")
print(f"\nDataset used: {X_heart_train.shape[0]} training samples")
print("Algorithm: Random Forest (200 trees)")

In [ ]:
# CELL 22: Evaluate Heart Disease model

y_heart_pred = model_heart.predict(X_heart_test)

acc_heart = accuracy_score(y_heart_test, y_heart_pred)
print(f"🎯 HEART DISEASE MODEL ACCURACY: {acc_heart * 100:.2f}%")
print("\n" + "="*50)
print("DETAILED CLASSIFICATION REPORT:")
print("="*50)
print(classification_report(y_heart_test, y_heart_pred,
      target_names=['No Heart Disease', 'Has Heart Disease']))

In [ ]:
# CELL 23: Confusion Matrix for Heart Disease

cm_heart = confusion_matrix(y_heart_test, y_heart_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm_heart, annot=True, fmt='d', cmap='Reds',
            xticklabels=['No Heart Disease', 'Has Heart Disease'],
            yticklabels=['No Heart Disease', 'Has Heart Disease'])
plt.title('Confusion Matrix — Heart Disease Model', fontsize=14)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.tight_layout()
plt.savefig('heart_confusion_matrix.png', dpi=150)
plt.show()

tn, fp, fn, tp = cm_heart.ravel()
print(f"\n📊 Results Breakdown:")
print(f"  ✅ Correctly predicted NO heart disease:  {tn}")
print(f"  ✅ Correctly predicted HAS heart disease: {tp}")
print(f"  ❌ False alarms (said yes, actually no): {fp}")
print(f"  ❌ Missed actual cases (said no, was yes): {fn}")

In [ ]:
# CELL 24: Feature Importance for Heart Disease

feat_heart_df = pd.DataFrame({
    'Feature': X_heart.columns,
    'Importance': model_heart.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(9, 6))
colors = ['#d73027' if x > 0.08 else '#4575b4' for x in feat_heart_df['Importance']]
plt.barh(feat_heart_df['Feature'], feat_heart_df['Importance'],
         color=colors, edgecolor='black')
plt.title('Feature Importance — Heart Disease Model\n(Red = Most Important)', fontsize=13)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('heart_feature_importance.png', dpi=150)
plt.show()

print("✅ Top 3 most important features for predicting Heart Disease:")
top3 = feat_heart_df.tail(3).iloc[::-1]
for _, row in top3.iterrows():
    print(f"   🔴 {row['Feature']}: {row['Importance']:.4f}")

In [ ]:
# CELL 25: Save Heart Disease model and scaler

joblib.dump(model_heart,  'heart_model.pkl')
joblib.dump(scaler_heart, 'heart_scaler.pkl')

print("✅ Model saved:  heart_model.pkl")
print("✅ Scaler saved: heart_scaler.pkl")
print("\n📌 Download both files from the Files panel (left sidebar)!")

In [ ]:
# CELL 26: Predict for a new heart disease patient
# ─────────────────────────────────────────────────────

sample_heart = np.array([[
    55,    # age
    1,     # sex: Male
    0,     # chest_pain_type (check Cell 18 output)
    130,   # resting_blood_pressure
    250,   # cholestoral
    1,     # fasting_blood_sugar: Lower than 120
    1,     # rest_ecg: Normal
    160,   # Max_heart_rate
    0,     # exercise_induced_angina: No
    1.5,   # oldpeak
    2,     # slope: Upsloping
    3,     # vessels_colored_by_flourosopy: Zero
    2      # thalassemia: Reversable Defect
]])

sample_heart_scaled = scaler_heart.transform(sample_heart)
prediction_heart = model_heart.predict(sample_heart_scaled)
probability_heart = model_heart.predict_proba(sample_heart_scaled)

print("="*45)
print("   HEART DISEASE PREDICTION RESULT")
print("="*45)
if prediction_heart[0] == 1:
    print("⚠️  Result: HIGH RISK of Heart Disease")
else:
    print("✅  Result: LOW RISK of Heart Disease")

print(f"\nConfidence:")
print(f"  No Heart Disease:  {probability_heart[0][0]*100:.1f}%")
print(f"  Has Heart Disease: {probability_heart[0][1]*100:.1f}%")

In [ ]:
# CELL 27: Load Kidney Disease dataset

df_kidney = pd.read_csv('kidney_disease.csv')

print("✅ Dataset loaded!")
print(f"Rows: {df_kidney.shape[0]},  Columns: {df_kidney.shape[1]}")
print("\nColumn names:")
print(df_kidney.columns.tolist())
print("\nFirst 5 rows:")
df_kidney.head()

In [ ]:
# CELL 28: Deep exploration of Kidney dataset

print("=== DATA TYPES ===")
print(df_kidney.dtypes)

print("\n=== MISSING VALUES (sorted) ===")
missing = df_kidney.isnull().sum()
missing_pct = (missing / len(df_kidney) * 100).round(1)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)
print(missing_df[missing_df['Missing Count'] > 0])

print("\n=== TARGET COLUMN ===")
print(df_kidney['classification'].value_counts())
print("\nckd = Chronic Kidney Disease,  notckd = No Kidney Disease")

In [ ]:
# CELL 29: Visualize missing values as a heatmap

plt.figure(figsize=(14, 5))
sns.heatmap(df_kidney.isnull(),
            cbar=False,
            cmap='viridis',
            yticklabels=False)
plt.title('Missing Values Map — Kidney Disease Dataset\n(Yellow = Missing, Purple = Present)',
          fontsize=13)
plt.tight_layout()
plt.savefig('kidney_missing_values.png', dpi=150)
plt.show()
print("✅ Missing values map saved!")
print("\nYellow streaks = columns with lots of missing data")

In [ ]:
# CELL 30: Drop the 'id' column (just a row number, not useful for prediction)

df_kidney_clean = df_kidney.copy()   
df_kidney_clean.drop('id', axis=1, inplace=True)

print("✅ Dropped 'id' column")
print(f"Shape now: {df_kidney_clean.shape}")
print("\nRemaining columns:")
print(df_kidney_clean.columns.tolist())

In [ ]:
# CELL 31: Clean and encode target column


df_kidney_clean['classification'] = df_kidney_clean['classification'].str.strip()


print("Unique target values BEFORE:", df_kidney_clean['classification'].unique())

df_kidney_clean['classification'] = df_kidney_clean['classification'].map({
    'ckd':    1,
    'notckd': 0
})

print("Unique target values AFTER:", df_kidney_clean['classification'].unique())
print("\nTarget distribution:")
print(df_kidney_clean['classification'].value_counts())
print("\n1 = Has Kidney Disease,  0 = No Kidney Disease")
print("\n✅ Target column fixed!")

In [ ]:
# CELL 32: Some numeric columns were read as text — fix them

numeric_cols = ['pcv', 'wc', 'rc']

for col in numeric_cols:
    df_kidney_clean[col] = pd.to_numeric(df_kidney_clean[col], errors='coerce')
    print(f"✅ Fixed '{col}' → now numeric, missing: {df_kidney_clean[col].isnull().sum()}")

print("\n✅ Numeric columns fixed!")

In [ ]:
# CELL 33: Fill missing values intelligently

print("Filling missing values...\n")

for col in df_kidney_clean.columns:
    missing_count = df_kidney_clean[col].isnull().sum()

    if missing_count == 0:
        continue   

    if df_kidney_clean[col].dtype == 'object':
        
        fill_val = df_kidney_clean[col].mode()[0]
        fill_type = "mode (most common)"
    else:
        
        fill_val = df_kidney_clean[col].median()
        fill_type = "median"

    df_kidney_clean[col].fillna(fill_val, inplace=True)
    print(f"  {col:10s} → filled {missing_count} gaps with {fill_type}: '{fill_val}'")

print(f"\n✅ All missing values filled!")
print(f"Total missing values remaining: {df_kidney_clean.isnull().sum().sum()}")

In [ ]:
# CELL 34: Encode all remaining text columns to numbers

encoders_kidney = {}

text_cols_kidney = df_kidney_clean.select_dtypes(include='object').columns.tolist()
# Remove target from list if it's there
text_cols_kidney = [c for c in text_cols_kidney if c != 'classification']

print(f"Text columns to encode: {text_cols_kidney}\n")

for col in text_cols_kidney:
    le = LabelEncoder()
    df_kidney_clean[col] = le.fit_transform(df_kidney_clean[col])
    encoders_kidney[col] = le
    print(f"✅ {col}: {list(le.classes_)} → {list(range(len(le.classes_)))}")

print("\n✅ All text columns encoded!")
print("\nFinal data types:")
print(df_kidney_clean.dtypes)

In [ ]:

# CELL 35: Final check before training

print("=== FINAL DATA CHECK ===")
print(f"Shape: {df_kidney_clean.shape}")
print(f"Missing values: {df_kidney_clean.isnull().sum().sum()}")
print(f"Data types — any object left: {(df_kidney_clean.dtypes == 'object').sum()}")

print("\n=== TARGET DISTRIBUTION ===")
print(df_kidney_clean['classification'].value_counts())

print("\n=== SAMPLE OF CLEAN DATA ===")
df_kidney_clean.head(3)

In [ ]:
# CELL 36: Visualize Kidney Disease data

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Chart 1: Target distribution
df_kidney_clean['classification'].value_counts().plot(
    kind='bar', ax=axes[0],
    color=['steelblue', 'darkorange'], edgecolor='black')
axes[0].set_title('Kidney Disease Distribution\n(0=No, 1=Yes)', fontsize=13)
axes[0].set_xticklabels(['No CKD', 'CKD'], rotation=0)
axes[0].set_ylabel('Count')

# Chart 2: Age distribution
df_kidney_clean['age'].plot(kind='hist', ax=axes[1], bins=20,
    color='darkorange', edgecolor='black')
axes[1].set_title('Age Distribution of Patients', fontsize=13)
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Count')

# Chart 3: Hemoglobin vs Blood Glucose colored by disease
scatter = axes[2].scatter(
    df_kidney_clean['hemo'],
    df_kidney_clean['bgr'],
    c=df_kidney_clean['classification'],
    cmap='RdYlGn', alpha=0.7, s=50, edgecolors='k', linewidths=0.3
)
axes[2].set_title('Hemoglobin vs Blood Glucose', fontsize=13)
axes[2].set_xlabel('Hemoglobin')
axes[2].set_ylabel('Blood Glucose')
plt.colorbar(scatter, ax=axes[2], label='Kidney Disease')

plt.tight_layout()
plt.savefig('kidney_exploration.png', dpi=150)
plt.show()
print("✅ Charts saved!")

In [ ]:
# CELL 37: Separate X and y

X_kidney = df_kidney_clean.drop('classification', axis=1)
y_kidney = df_kidney_clean['classification']

print("Feature columns:")
for i, col in enumerate(X_kidney.columns):
    print(f"  {i+1:2d}. {col}")

print(f"\nTotal features: {X_kidney.shape[1]}")
print(f"Total samples:  {X_kidney.shape[0]}")

# Scale features
scaler_kidney = StandardScaler()
X_kidney_scaled = scaler_kidney.fit_transform(X_kidney)

print("\n✅ Features prepared and scaled!")

In [ ]:
# CELL 38: Train/Test split

X_kidney_train, X_kidney_test, y_kidney_train, y_kidney_test = train_test_split(
    X_kidney_scaled, y_kidney,
    test_size=0.2,
    random_state=42
)

print(f"Training samples:  {X_kidney_train.shape[0]}")
print(f"Testing samples:   {X_kidney_test.shape[0]}")
print("\n✅ Split done! (80% train / 20% test)")

In [ ]:
# CELL 39: Train model for Kidney Disease

print("⏳ Training Kidney Disease model...")

model_kidney = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=4,   # extra guard against overfitting on small dataset
    random_state=42
)

model_kidney.fit(X_kidney_train, y_kidney_train)

print("✅ Model trained successfully!")
print(f"\nTrained on: {X_kidney_train.shape[0]} samples")
print(f"Features used: {X_kidney_train.shape[1]}")

In [ ]:
# CELL 40: Evaluate Kidney Disease model

y_kidney_pred = model_kidney.predict(X_kidney_test)

acc_kidney = accuracy_score(y_kidney_test, y_kidney_pred)
print(f"🎯 KIDNEY DISEASE MODEL ACCURACY: {acc_kidney * 100:.2f}%")
print("\n" + "="*50)
print("DETAILED CLASSIFICATION REPORT:")
print("="*50)
print(classification_report(y_kidney_test, y_kidney_pred,
      target_names=['No Kidney Disease', 'Has Kidney Disease']))

In [ ]:
# CELL 41: Confusion Matrix for Kidney Disease

cm_kidney = confusion_matrix(y_kidney_test, y_kidney_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm_kidney, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['No Kidney Disease', 'Has Kidney Disease'],
            yticklabels=['No Kidney Disease', 'Has Kidney Disease'])
plt.title('Confusion Matrix — Kidney Disease Model', fontsize=14)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.tight_layout()
plt.savefig('kidney_confusion_matrix.png', dpi=150)
plt.show()

tn, fp, fn, tp = cm_kidney.ravel()
print(f"\n📊 Results Breakdown:")
print(f"  ✅ Correctly predicted NO kidney disease:  {tn}")
print(f"  ✅ Correctly predicted HAS kidney disease: {tp}")
print(f"  ❌ False alarms (said yes, actually no):  {fp}")
print(f"  ❌ Missed actual cases (said no, was yes): {fn}")

In [ ]:
# CELL 42: Feature Importance for Kidney Disease

feat_kidney_df = pd.DataFrame({
    'Feature': X_kidney.columns,
    'Importance': model_kidney.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 8))
colors = ['#e34a33' if x > 0.06 else '#fdbb84' if x > 0.03 else '#fee8c8'
          for x in feat_kidney_df['Importance']]
plt.barh(feat_kidney_df['Feature'], feat_kidney_df['Importance'],
         color=colors, edgecolor='black')
plt.title('Feature Importance — Kidney Disease Model\n(Dark Red = Most Important)',
          fontsize=13)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('kidney_feature_importance.png', dpi=150)
plt.show()

print("✅ Top 5 most important features for predicting Kidney Disease:")
top5 = feat_kidney_df.tail(5).iloc[::-1]
for _, row in top5.iterrows():
    print(f"   🔴 {row['Feature']}: {row['Importance']:.4f}")

In [ ]:
# CELL 43: Save Kidney Disease model and scaler

joblib.dump(model_kidney,  'kidney_model.pkl')
joblib.dump(scaler_kidney, 'kidney_scaler.pkl')

print("✅ Model saved:  kidney_model.pkl")
print("✅ Scaler saved: kidney_scaler.pkl")
print("\n📌 Download both files from the Files panel (left sidebar)!")

In [ ]:
# CELL 44: Predict for a new kidney disease patient
# ─────────────────────────────────────────────────────────────────

sample_kidney = np.array([[
    48,     # age
    80,     # bp (blood pressure)
    1.02,   # sg (specific gravity)
    1,      # al (albumin)
    0,      # su (sugar)
    1,      # rbc: normal
    1,      # pc: normal
    0,      # pcc: notpresent
    0,      # ba: notpresent
    121,    # bgr (blood glucose random)
    36,     # bu (blood urea)
    1.2,    # sc (serum creatinine)
    138,    # sod (sodium)
    4.5,    # pot (potassium)
    15.4,   # hemo (hemoglobin)
    44,     # pcv (packed cell volume)
    7800,   # wc (white blood cell count)
    5.2,    # rc (red blood cell count)
    1,      # htn: yes
    1,      # dm: yes
    0,      # cad: no
    0,      # appet: good
    0,      # pe: no
    0       # ane: no
]])

sample_kidney_scaled = scaler_kidney.transform(sample_kidney)
prediction_kidney = model_kidney.predict(sample_kidney_scaled)
probability_kidney = model_kidney.predict_proba(sample_kidney_scaled)

print("="*45)
print("   KIDNEY DISEASE PREDICTION RESULT")
print("="*45)
if prediction_kidney[0] == 1:
    print("⚠️  Result: HIGH RISK of Kidney Disease (CKD)")
else:
    print("✅  Result: LOW RISK of Kidney Disease")

print(f"\nConfidence:")
print(f"  No Kidney Disease:  {probability_kidney[0][0]*100:.1f}%")
print(f"  Has Kidney Disease: {probability_kidney[0][1]*100:.1f}%")

In [ ]:
# CELL 45: Load Liver Disease dataset

df_liver = pd.read_csv('Indian Liver Patient Dataset.csv')

print("✅ Dataset loaded!")
print(f"Rows: {df_liver.shape[0]},  Columns: {df_liver.shape[1]}")
print("\nColumn names:")
print(df_liver.columns.tolist())
print("\nFirst 5 rows:")
df_liver.head()

In [ ]:
# CELL 46: Explore Liver Disease data

print("=== DATA TYPES ===")
print(df_liver.dtypes)

print("\n=== MISSING VALUES ===")
missing = df_liver.isnull().sum()
print(missing[missing > 0])

print("\n=== TARGET COLUMN (is_patient) ===")
print(df_liver['is_patient'].value_counts())
print("\n1 = Has Liver Disease,  2 = No Liver Disease")

print("\n=== GENDER COLUMN ===")
print(df_liver['gender'].value_counts())

print("\n=== BASIC STATISTICS ===")
df_liver.describe()

In [ ]:
# CELL 47: Visualize Liver Disease data

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Chart 1: Target distribution
df_liver['is_patient'].value_counts().plot(
    kind='bar', ax=axes[0],
    color=['tomato', 'mediumseagreen'], edgecolor='black')
axes[0].set_title('Liver Disease Distribution\n(1=Patient, 2=Not Patient)', fontsize=13)
axes[0].set_xticklabels(['Has Disease', 'No Disease'], rotation=0)
axes[0].set_ylabel('Count')

# Chart 2: Age distribution by disease
df_liver[df_liver['is_patient'] == 1]['age'].plot(
    kind='hist', ax=axes[1], bins=20,
    color='tomato', alpha=0.6, label='Has Disease', edgecolor='black')
df_liver[df_liver['is_patient'] == 2]['age'].plot(
    kind='hist', ax=axes[1], bins=20,
    color='mediumseagreen', alpha=0.6, label='No Disease', edgecolor='black')
axes[1].set_title('Age Distribution by Disease Status', fontsize=13)
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Count')
axes[1].legend()

# Chart 3: Total Bilirubin vs SGPT colored by disease
scatter = axes[2].scatter(
    df_liver['tot_bilirubin'],
    df_liver['sgpt'],
    c=df_liver['is_patient'],
    cmap='RdYlGn_r', alpha=0.6, s=40,
    edgecolors='k', linewidths=0.3
)
axes[2].set_title('Total Bilirubin vs SGPT', fontsize=13)
axes[2].set_xlabel('Total Bilirubin')
axes[2].set_ylabel('SGPT')
plt.colorbar(scatter, ax=axes[2], label='1=Disease, 2=No Disease')

plt.tight_layout()
plt.savefig('liver_exploration.png', dpi=150)
plt.show()
print("✅ Charts saved!")

In [ ]:
# CELL 48: Convert target from (1, 2) to (1, 0)
# Currently: 1 = has disease, 2 = no disease
# We want:   1 = has disease, 0 = no disease  (standard ML format)

df_liver_clean = df_liver.copy()

df_liver_clean['is_patient'] = df_liver_clean['is_patient'].map({
    1: 1,   # has liver disease → 1
    2: 0    # no liver disease  → 0
})

print("Target values AFTER fix:")
print(df_liver_clean['is_patient'].value_counts())
print("\n1 = Has Liver Disease")
print("0 = No Liver Disease")
print("\n✅ Target column fixed!")

In [ ]:
# CELL 49: Encode gender (only text column in this dataset)

le_liver = LabelEncoder()
df_liver_clean['gender'] = le_liver.fit_transform(df_liver_clean['gender'])

print("Gender encoding:")
for i, cls in enumerate(le_liver.classes_):
    print(f"  {cls} → {i}")

print("\n✅ Gender encoded!")
print("\nData types now:")
print(df_liver_clean.dtypes)

In [ ]:
# CELL 50: Fill the 4 missing values in ag_ratio column

print(f"Missing in ag_ratio BEFORE: {df_liver_clean['ag_ratio'].isnull().sum()}")

df_liver_clean['ag_ratio'].fillna(
    df_liver_clean['ag_ratio'].median(), inplace=True
)

print(f"Missing in ag_ratio AFTER:  {df_liver_clean['ag_ratio'].isnull().sum()}")
print(f"\nTotal missing values: {df_liver_clean.isnull().sum().sum()}")
print("\n✅ All missing values filled!")

In [ ]:
# CELL 51: Final check before training

print("=== FINAL DATA CHECK ===")
print(f"Shape:          {df_liver_clean.shape}")
print(f"Missing values: {df_liver_clean.isnull().sum().sum()}")
print(f"Object columns: {(df_liver_clean.dtypes == 'object').sum()}")

print("\n=== TARGET DISTRIBUTION ===")
print(df_liver_clean['is_patient'].value_counts())

print("\n=== CLEAN DATA SAMPLE ===")
df_liver_clean.head(3)

In [ ]:
# CELL 52: Separate X and y

X_liver = df_liver_clean.drop('is_patient', axis=1)
y_liver = df_liver_clean['is_patient']

print("Feature columns:")
for i, col in enumerate(X_liver.columns):
    print(f"  {i+1:2d}. {col}")

print(f"\nTotal features: {X_liver.shape[1]}")
print(f"Total samples:  {X_liver.shape[0]}")

# Scale features
scaler_liver = StandardScaler()
X_liver_scaled = scaler_liver.fit_transform(X_liver)

print("\n✅ Features prepared and scaled!")

In [ ]:
# CELL 53: Train/Test split

X_liver_train, X_liver_test, y_liver_train, y_liver_test = train_test_split(
    X_liver_scaled, y_liver,
    test_size=0.2,
    random_state=42
)

print(f"Training samples:  {X_liver_train.shape[0]}")
print(f"Testing samples:   {X_liver_test.shape[0]}")
print("\n✅ Split done! (80% train / 20% test)")

In [ ]:
# CELL 54: Train Random Forest model for Liver Disease

print("⏳ Training Liver Disease model...")

model_liver = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_split=5,   
    min_samples_leaf=2,    
    random_state=42
)

model_liver.fit(X_liver_train, y_liver_train)

print("✅ Model trained successfully!")
print(f"\nTrained on: {X_liver_train.shape[0]} samples")
print(f"Features used: {X_liver_train.shape[1]}")

In [ ]:
# CELL 55: Evaluate Liver Disease model

y_liver_pred = model_liver.predict(X_liver_test)

acc_liver = accuracy_score(y_liver_test, y_liver_pred)
print(f"🎯 LIVER DISEASE MODEL ACCURACY: {acc_liver * 100:.2f}%")
print("\n" + "="*50)
print("DETAILED CLASSIFICATION REPORT:")
print("="*50)
print(classification_report(y_liver_test, y_liver_pred,
      target_names=['No Liver Disease', 'Has Liver Disease']))

In [ ]:
# CELL 56: Confusion Matrix for Liver Disease

cm_liver = confusion_matrix(y_liver_test, y_liver_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm_liver, annot=True, fmt='d', cmap='Greens',
            xticklabels=['No Liver Disease', 'Has Liver Disease'],
            yticklabels=['No Liver Disease', 'Has Liver Disease'])
plt.title('Confusion Matrix — Liver Disease Model', fontsize=14)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.tight_layout()
plt.savefig('liver_confusion_matrix.png', dpi=150)
plt.show()

tn, fp, fn, tp = cm_liver.ravel()
print(f"\n📊 Results Breakdown:")
print(f"  ✅ Correctly predicted NO liver disease:  {tn}")
print(f"  ✅ Correctly predicted HAS liver disease: {tp}")
print(f"  ❌ False alarms (said yes, actually no):  {fp}")
print(f"  ❌ Missed actual cases (said no, was yes): {fn}")

In [ ]:
# CELL 57: Feature Importance for Liver Disease

feat_liver_df = pd.DataFrame({
    'Feature': X_liver.columns,
    'Importance': model_liver.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(9, 6))
colors = ['#1a9641' if x > 0.12 else '#a6d96a' if x > 0.07 else '#d9ef8b'
          for x in feat_liver_df['Importance']]
plt.barh(feat_liver_df['Feature'], feat_liver_df['Importance'],
         color=colors, edgecolor='black')
plt.title('Feature Importance — Liver Disease Model\n(Dark Green = Most Important)',
          fontsize=13)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('liver_feature_importance.png', dpi=150)
plt.show()

print("✅ Top 3 most important features for predicting Liver Disease:")
top3 = feat_liver_df.tail(3).iloc[::-1]
for _, row in top3.iterrows():
    print(f"   🟢 {row['Feature']}: {row['Importance']:.4f}")

In [ ]:
# CELL 58: Save Liver Disease model and scaler

joblib.dump(model_liver,  'liver_model.pkl')
joblib.dump(scaler_liver, 'liver_scaler.pkl')

print("✅ Model saved:  liver_model.pkl")
print("✅ Scaler saved: liver_scaler.pkl")
print("\n📌 Download both files from the Files panel (left sidebar)!")

In [ ]:
# CELL 59: Predict for a new liver disease patient


sample_liver = np.array([[
    45,     # age
    1,      # gender: Male
    1.2,    # tot_bilirubin
    0.4,    # direct_bilirubin
    220,    # tot_proteins
    34,     # albumin
    1.1,    # ag_ratio
    56,     # sgpt
    40,     # sgot
    290     # alkphos
]])

sample_liver_scaled = scaler_liver.transform(sample_liver)
prediction_liver = model_liver.predict(sample_liver_scaled)
probability_liver = model_liver.predict_proba(sample_liver_scaled)

print("="*45)
print("   LIVER DISEASE PREDICTION RESULT")
print("="*45)
if prediction_liver[0] == 1:
    print("⚠️  Result: HIGH RISK of Liver Disease")
else:
    print("✅  Result: LOW RISK of Liver Disease")

print(f"\nConfidence:")
print(f"  No Liver Disease:  {probability_liver[0][0]*100:.1f}%")
print(f"  Has Liver Disease: {probability_liver[0][1]*100:.1f}%")

In [ ]:
# CELL 60: Final summary of all 4 models

print("=" * 55)
print("   🏆  ALL 4 DISEASE MODELS — FINAL SUMMARY")
print("=" * 55)

results = {
    "Diabetes":      accuracy_score(y_test,          model.predict(X_test)),
    "Heart Disease": accuracy_score(y_heart_test,    model_heart.predict(X_heart_test)),
    "Kidney Disease":accuracy_score(y_kidney_test,   model_kidney.predict(X_kidney_test)),
    "Liver Disease": accuracy_score(y_liver_test,    model_liver.predict(X_liver_test))
}

for disease, acc in results.items():
    bar = "█" * int(acc * 30)
    print(f"  {disease:<18} {acc*100:5.2f}%  {bar}")

print("=" * 55)
print("\n📦 Saved model files:")
saved = [
    'diabetes_model.pkl',  'diabetes_scaler.pkl',
    'heart_model.pkl',     'heart_scaler.pkl',
    'kidney_model.pkl',    'kidney_scaler.pkl',
    'liver_model.pkl',     'liver_scaler.pkl'
]
for f in saved:
    print(f"  ✅ {f}")

print("\n🎉 Phase 3 COMPLETE! All 4 models trained and saved!")
print("📌 Make sure all 8 .pkl files are downloaded to your computer.")

In [ ]:
# CELL 61: MASTER VERIFICATION — Test all 4 saved models

import joblib
import numpy as np

print("=" * 60)
print("   🔍  LOADING & TESTING ALL 4 SAVED MODELS")
print("=" * 60)

# ── Load all 8 files ──────────────────────────────────────────
try:
    m_diabetes  = joblib.load('diabetes_model.pkl')
    s_diabetes  = joblib.load('diabetes_scaler.pkl')
    print("✅ Diabetes   model + scaler loaded")
except:
    print("❌ Diabetes   model MISSING — re-run Cell 13")

try:
    m_heart     = joblib.load('heart_model.pkl')
    s_heart     = joblib.load('heart_scaler.pkl')
    print("✅ Heart      model + scaler loaded")
except:
    print("❌ Heart      model MISSING — re-run Cell 25")

try:
    m_kidney    = joblib.load('kidney_model.pkl')
    s_kidney    = joblib.load('kidney_scaler.pkl')
    print("✅ Kidney     model + scaler loaded")
except:
    print("❌ Kidney     model MISSING — re-run Cell 43")

try:
    m_liver     = joblib.load('liver_model.pkl')
    s_liver     = joblib.load('liver_scaler.pkl')
    print("✅ Liver      model + scaler loaded")
except:
    print("❌ Liver      model MISSING — re-run Cell 58")

In [ ]:
# CELL 62: Run a live prediction through every model

print("\n" + "=" * 60)
print("   🧪  LIVE PREDICTION TEST — ALL 4 MODELS")
print("=" * 60)

results_check = {}

# ── 1. DIABETES ───────────────────────────────────────────────
try:
    t_diabetes = np.array([[1, 45, 1, 0, 0, 28.5, 6.5, 140]])
    pred = m_diabetes.predict(s_diabetes.transform(t_diabetes))[0]
    prob = m_diabetes.predict_proba(s_diabetes.transform(t_diabetes))[0]
    label = "⚠️ HIGH RISK" if pred == 1 else "✅ LOW RISK"
    results_check['Diabetes'] = f"{label}  (confidence: {max(prob)*100:.1f}%)"
    print(f"\n🩸 Diabetes      → {results_check['Diabetes']}")
except Exception as e:
    print(f"\n❌ Diabetes test failed: {e}")

# ── 2. HEART DISEASE ──────────────────────────────────────────
try:
    t_heart = np.array([[55, 1, 0, 130, 250, 1, 1, 160, 0, 1.5, 2, 3, 2]])
    pred = m_heart.predict(s_heart.transform(t_heart))[0]
    prob = m_heart.predict_proba(s_heart.transform(t_heart))[0]
    label = "⚠️ HIGH RISK" if pred == 1 else "✅ LOW RISK"
    results_check['Heart'] = f"{label}  (confidence: {max(prob)*100:.1f}%)"
    print(f"❤️  Heart Disease → {results_check['Heart']}")
except Exception as e:
    print(f"\n❌ Heart test failed: {e}")

# ── 3. KIDNEY DISEASE ─────────────────────────────────────────
try:
    t_kidney = np.array([[48, 80, 1.02, 1, 0, 1, 1, 0, 0,
                          121, 36, 1.2, 138, 4.5, 15.4, 44, 7800, 5.2,
                          1, 1, 0, 0, 0, 0]])
    pred = m_kidney.predict(s_kidney.transform(t_kidney))[0]
    prob = m_kidney.predict_proba(s_kidney.transform(t_kidney))[0]
    label = "⚠️ HIGH RISK" if pred == 1 else "✅ LOW RISK"
    results_check['Kidney'] = f"{label}  (confidence: {max(prob)*100:.1f}%)"
    print(f"🫘 Kidney Disease → {results_check['Kidney']}")
except Exception as e:
    print(f"\n❌ Kidney test failed: {e}")

# ── 4. LIVER DISEASE ──────────────────────────────────────────
try:
    t_liver = np.array([[45, 1, 1.2, 0.4, 220, 34, 1.1, 56, 40, 290]])
    pred = m_liver.predict(s_liver.transform(t_liver))[0]
    prob = m_liver.predict_proba(s_liver.transform(t_liver))[0]
    label = "⚠️ HIGH RISK" if pred == 1 else "✅ LOW RISK"
    results_check['Liver'] = f"{label}  (confidence: {max(prob)*100:.1f}%)"
    print(f"🫀 Liver Disease  → {results_check['Liver']}")
except Exception as e:
    print(f"\n❌ Liver test failed: {e}")

In [ ]:
# CELL 63: Final confirmation report

print("\n" + "=" * 60)
print("   📊  ACCURACY SCORES — ALL 4 MODELS")
print("=" * 60)

from sklearn.metrics import accuracy_score

scores = {
    "🩸 Diabetes     ": accuracy_score(y_test,        model.predict(X_test)),
    "❤️  Heart Disease": accuracy_score(y_heart_test,  model_heart.predict(X_heart_test)),
    "🫘 Kidney Disease": accuracy_score(y_kidney_test, model_kidney.predict(X_kidney_test)),
    "🫀 Liver Disease ": accuracy_score(y_liver_test,  model_liver.predict(X_liver_test)),
}

all_passed = True
for name, acc in scores.items():
    bar   = "█" * int(acc * 40)
    flag  = "✅" if acc >= 0.70 else "⚠️"
    print(f"  {flag} {name}  {acc*100:6.2f}%  {bar}")
    if acc < 0.70:
        all_passed = False

print("\n" + "=" * 60)

if all_passed:
    print("""
  🎉🎉🎉  ALL 4 MODELS VERIFIED SUCCESSFULLY! 🎉🎉🎉

  ✅ All models load correctly from .pkl files
  ✅ All models make predictions without errors
  ✅ All accuracy scores are above threshold
  ✅ Ready to build the Web App!

  📌 REMINDER — Make sure these 8 files are downloaded:
     • diabetes_model.pkl   • diabetes_scaler.pkl
     • heart_model.pkl      • heart_scaler.pkl
     • kidney_model.pkl     • kidney_scaler.pkl
     • liver_model.pkl      • liver_scaler.pkl
""")
else:
    print("""
  ⚠️  One or more models need attention.
  Check the ⚠️ flagged model above and re-run its training cells.
""")

In [ ]:
# CELL 64: Install streamlit and tunnel tool

!pip install streamlit pyngrok -q

print("✅ Streamlit and pyngrok installed!")

In [ ]:
# CELL 69: Install SHAP library

!pip install shap -q

import shap
print("✅ SHAP installed successfully!")
print(f"   SHAP version: {shap.__version__}")

In [ ]:
# CELL 70: Create SHAP explainers for all 4 models

import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Building SHAP explainers for all 4 models...\n")

# ── Diabetes ──────────────────────────────────────────────────
explainer_diabetes = shap.TreeExplainer(model)
print("✅ Diabetes  explainer ready")

# ── Heart ─────────────────────────────────────────────────────
explainer_heart = shap.TreeExplainer(model_heart)
print("✅ Heart     explainer ready")

# ── Kidney ────────────────────────────────────────────────────
explainer_kidney = shap.TreeExplainer(model_kidney)
print("✅ Kidney    explainer ready")

# ── Liver ─────────────────────────────────────────────────────
explainer_liver = shap.TreeExplainer(model_liver)
print("✅ Liver     explainer ready")

print("\n🎉 All 4 SHAP explainers are ready!")

In [ ]:
# CELL 71: Test SHAP explanation on Diabetes sample

import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Feature names for diabetes
diabetes_features = ['gender','age','hypertension','heart_disease',
                     'smoking_history','bmi','HbA1c_level','blood_glucose_level']

# Sample high-risk patient
sample = np.array([[1, 58, 1, 1, 1, 35.0, 8.5, 220]])

# Scale it
sample_scaled = scaler.transform(sample)

# Get SHAP values
shap_values = explainer_diabetes.shap_values(sample_scaled)


if isinstance(shap_values, list):
    # list format → [class0_array, class1_array]
    sv = np.array(shap_values[1]).flatten()
elif isinstance(shap_values, np.ndarray):
    if shap_values.ndim == 3:
        # shape (1, n_features, 2) → take class1
        sv = shap_values[0, :, 1].flatten()
    elif shap_values.ndim == 2:
        # shape (1, n_features)
        sv = shap_values[0].flatten()
    else:
        sv = shap_values.flatten()
else:
    sv = np.array(shap_values).flatten()

print(f"SHAP values shape fixed → {sv.shape}")
print(f"Number of features      → {len(diabetes_features)}")

# ── Build explanation dataframe ───────────────────────────────
explanation_df = pd.DataFrame({
    'Feature'   : diabetes_features,
    'Value'     : sample[0].tolist(),     # ← .tolist() ensures 1D flat list
    'SHAP Score': sv.tolist()             # ← .tolist() ensures 1D flat list
}).sort_values('SHAP Score', key=abs, ascending=True)

print("\nSHAP Explanation Table:")
print(explanation_df.to_string(index=False))

# ── Plot ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#d73027' if x > 0 else '#4575b4' for x in explanation_df['SHAP Score']]
bars = ax.barh(explanation_df['Feature'],
               explanation_df['SHAP Score'],
               color=colors, edgecolor='black')

ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('🧠 SHAP Explanation — Why this Diabetes prediction?\n'
             '(Red = Increases Risk  |  Blue = Decreases Risk)', fontsize=12)
ax.set_xlabel('SHAP Value (Impact on Prediction)')

# Add value labels on bars
for bar, val, feat_val in zip(bars,
                               explanation_df['SHAP Score'],
                               explanation_df['Value']):
    label_x  = bar.get_width() + 0.001 if val >= 0 else bar.get_width() - 0.001
    label_ha = 'left' if val >= 0 else 'right'
    ax.text(label_x,
            bar.get_y() + bar.get_height() / 2,
            f'  val={feat_val}',
            va='center', fontsize=9, ha=label_ha)

# Legend
red_patch  = mpatches.Patch(color='#d73027', label='Increases Risk')
blue_patch = mpatches.Patch(color='#4575b4', label='Decreases Risk')
ax.legend(handles=[red_patch, blue_patch], fontsize=9)

plt.tight_layout()
plt.savefig('shap_diabetes_test.png', dpi=150)
plt.show()
print("\n✅ SHAP chart working perfectly!")

In [ ]:
app_code = r'''
import streamlit as st
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib
import shap
import warnings
warnings.filterwarnings("ignore")

st.set_page_config(
    page_title="Disease Prediction System",
    page_icon="🏥",
    layout="wide"
)

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');

* { font-family: 'Inter', sans-serif; }

.main { background-color: #f4f6f9; }

.stButton > button {
    width: 100%;
    background: linear-gradient(135deg, #1e3a5f, #2563a8);
    color: white;
    font-size: 16px;
    font-weight: 600;
    padding: 14px;
    border-radius: 10px;
    border: none;
    margin-top: 10px;
    letter-spacing: 0.5px;
    transition: all 0.2s;
}
.stButton > button:hover {
    background: linear-gradient(135deg, #16304f, #1a4f8a);
    box-shadow: 0 4px 15px rgba(37,99,168,0.4);
}

/* Input section card */
.input-card {
    background: white;
    border-radius: 14px;
    padding: 20px 24px;
    border: 1px solid #e2e8f0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.06);
    margin-bottom: 16px;
}

/* Flow arrow */
.flow-arrow {
    text-align: center;
    font-size: 28px;
    color: #64748b;
    margin: 6px 0;
}

/* AI processing box */
.ai-box {
    background: linear-gradient(135deg, #1e3a5f, #2d5a8e);
    color: white;
    border-radius: 14px;
    padding: 22px;
    text-align: center;
    margin: 10px 0;
    box-shadow: 0 4px 20px rgba(30,58,95,0.3);
}

/* Result box HIGH RISK */
.result-high {
    background: linear-gradient(135deg, #fff5f5, #ffe8e8);
    border: 2px solid #dc2626;
    border-radius: 14px;
    padding: 22px 28px;
    margin: 12px 0;
}
/* Result box LOW RISK */
.result-low {
    background: linear-gradient(135deg, #f0fdf4, #dcfce7);
    border: 2px solid #16a34a;
    border-radius: 14px;
    padding: 22px 28px;
    margin: 12px 0;
}

/* Factor row — risk (positive SHAP) */
.factor-risk {
    background: white;
    border-left: 5px solid #dc2626;
    border-radius: 0 10px 10px 0;
    padding: 12px 16px;
    margin: 6px 0;
    box-shadow: 0 1px 4px rgba(0,0,0,0.07);
    display: flex;
    align-items: center;
}
/* Factor row — protective (negative SHAP) */
.factor-safe {
    background: white;
    border-left: 5px solid #2563a8;
    border-radius: 0 10px 10px 0;
    padding: 12px 16px;
    margin: 6px 0;
    box-shadow: 0 1px 4px rgba(0,0,0,0.07);
}

/* Section title */
.section-title {
    font-size: 15px;
    font-weight: 700;
    color: #1e3a5f;
    text-transform: uppercase;
    letter-spacing: 1px;
    margin: 18px 0 10px 0;
    padding-bottom: 6px;
    border-bottom: 2px solid #e2e8f0;
}

/* Parameter pill */
.param-pill {
    display: inline-block;
    padding: 3px 10px;
    border-radius: 20px;
    font-size: 12px;
    font-weight: 600;
    margin-left: 8px;
}
.pill-normal   { background:#dcfce7; color:#15803d; }
.pill-warning  { background:#fef9c3; color:#a16207; }
.pill-critical { background:#fee2e2; color:#dc2626; }

/* Summary box */
.summary-box {
    background: #f8fafc;
    border: 1px solid #e2e8f0;
    border-radius: 12px;
    padding: 18px 22px;
    margin-top: 14px;
}
</style>
""", unsafe_allow_html=True)


# ══════════════════════════════════════════════════════════════
#   NORMAL RANGES
# ══════════════════════════════════════════════════════════════
NORMAL_RANGES = {
    "age"                     : (0,   120,  "yrs"),
    "bmi"                     : (18.5,24.9, "kg/m²"),
    "HbA1c_level"             : (4.0, 5.6,  "%"),
    "blood_glucose_level"     : (70,  99,   "mg/dL"),
    "resting_blood_pressure"  : (90,  120,  "mmHg"),
    "cholestoral"             : (0,   200,  "mg/dL"),
    "Max_heart_rate"          : (60,  100,  "bpm"),
    "oldpeak"                 : (0,   1.0,  "mm"),
    "bp"                      : (60,  80,   "mmHg"),
    "bgr"                     : (70,  140,  "mg/dL"),
    "bu"                      : (7,   20,   "mg/dL"),
    "sc"                      : (0.6, 1.2,  "mg/dL"),
    "sod"                     : (136, 145,  "mEq/L"),
    "pot"                     : (3.5, 5.0,  "mEq/L"),
    "hemo"                    : (12,  17,   "g/dL"),
    "pcv"                     : (36,  50,   "%"),
    "wc"                      : (4000,11000,"cells/µL"),
    "rc"                      : (4.2, 5.9,  "M/µL"),
    "tot_bilirubin"           : (0.1, 1.2,  "mg/dL"),
    "direct_bilirubin"        : (0.0, 0.3,  "mg/dL"),
    "tot_proteins"            : (6.3, 8.2,  "g/dL"),
    "albumin"                 : (3.5, 5.0,  "g/dL"),
    "ag_ratio"                : (1.0, 2.5,  "ratio"),
    "sgpt"                    : (7,   56,   "U/L"),
    "sgot"                    : (10,  40,   "U/L"),
    "alkphos"                 : (44,  147,  "U/L"),
}

LABELS = {
    "gender":"Gender","age":"Age","hypertension":"Hypertension",
    "heart_disease":"Heart Disease","smoking_history":"Smoking History",
    "bmi":"BMI","HbA1c_level":"HbA1c Level",
    "blood_glucose_level":"Blood Glucose",
    "sex":"Sex","chest_pain_type":"Chest Pain Type",
    "resting_blood_pressure":"Resting BP","cholestoral":"Cholesterol",
    "fasting_blood_sugar":"Fasting Blood Sugar","rest_ecg":"Resting ECG",
    "Max_heart_rate":"Max Heart Rate",
    "exercise_induced_angina":"Exercise Angina",
    "oldpeak":"ST Depression","slope":"ST Slope",
    "vessels_colored_by_flourosopy":"Vessels Colored",
    "thalassemia":"Thalassemia","bp":"Blood Pressure",
    "sg":"Specific Gravity","al":"Albumin Level","su":"Sugar Level",
    "rbc":"Red Blood Cells","pc":"Pus Cells","pcc":"Pus Cell Clumps",
    "ba":"Bacteria","bgr":"Blood Glucose (Random)","bu":"Blood Urea",
    "sc":"Serum Creatinine","sod":"Sodium","pot":"Potassium",
    "hemo":"Hemoglobin","pcv":"Packed Cell Volume",
    "wc":"White Blood Cells","rc":"Red Blood Cell Count",
    "htn":"Hypertension","dm":"Diabetes Mellitus",
    "cad":"Coronary Artery Disease","appet":"Appetite",
    "pe":"Pedal Edema","ane":"Anemia",
    "tot_bilirubin":"Total Bilirubin","direct_bilirubin":"Direct Bilirubin",
    "tot_proteins":"Total Proteins","albumin":"Albumin",
    "ag_ratio":"A/G Ratio","sgpt":"SGPT (ALT)","sgot":"SGOT (AST)",
    "alkphos":"Alkaline Phosphatase",
}

FEATURES = {
    "diabetes":["gender","age","hypertension","heart_disease",
                "smoking_history","bmi","HbA1c_level","blood_glucose_level"],
    "heart"   :["age","sex","chest_pain_type","resting_blood_pressure",
                "cholestoral","fasting_blood_sugar","rest_ecg",
                "Max_heart_rate","exercise_induced_angina","oldpeak",
                "slope","vessels_colored_by_flourosopy","thalassemia"],
    "kidney"  :["age","bp","sg","al","su","rbc","pc","pcc","ba",
                "bgr","bu","sc","sod","pot","hemo","pcv","wc","rc",
                "htn","dm","cad","appet","pe","ane"],
    "liver"   :["age","gender","tot_bilirubin","direct_bilirubin",
                "tot_proteins","albumin","ag_ratio","sgpt","sgot","alkphos"],
}

DISEASE_META = {
    "diabetes" : {"label":"Diabetes",       "icon":"🩸", "color":"#dc2626"},
    "heart"    : {"label":"Heart Disease",  "icon":"❤️", "color":"#dc2626"},
    "kidney"   : {"label":"Kidney Disease", "icon":"🫘", "color":"#dc2626"},
    "liver"    : {"label":"Liver Disease",  "icon":"🫀", "color":"#dc2626"},
}


# ══════════════════════════════════════════════════════════════
#   LOAD MODELS
# ══════════════════════════════════════════════════════════════
@st.cache_resource
def load_all_models():
    models, scalers, explainers = {}, {}, {}
    for d in ["diabetes","heart","kidney","liver"]:
        try:
            m = joblib.load(f"{d}_model.pkl")
            s = joblib.load(f"{d}_scaler.pkl")
            models[d]     = m
            scalers[d]    = s
            explainers[d] = shap.TreeExplainer(m)
        except Exception as e:
            st.error(f"Cannot load {d} model: {e}")
    return models, scalers, explainers

models, scalers, explainers = load_all_models()


# ══════════════════════════════════════════════════════════════
#   CORE HELPERS
# ══════════════════════════════════════════════════════════════
def safe_predict(key, feat):
    try:
        scaled = scalers[key].transform(feat)
        pred   = int(models[key].predict(scaled)[0])
        prob   = models[key].predict_proba(scaled)[0].tolist()
        return pred, prob
    except Exception as e:
        st.error(f"Prediction error: {e}")
        return 0, [0.5, 0.5]


def safe_shap(key, feat, n_features):
    try:
        scaled = scalers[key].transform(feat)
        raw    = explainers[key].shap_values(scaled)
        if isinstance(raw, list):
            sv = np.array(raw[1]).flatten()
        else:
            sv = np.array(raw).flatten()
        if len(sv) == 2 * n_features:
            sv = sv[n_features:]
        return sv[:n_features]
    except:
        return np.zeros(n_features)


def get_status(fname, value):
    if fname not in NORMAL_RANGES:
        return "info", "#64748b"
    lo, hi, _ = NORMAL_RANGES[fname]
    try:
        v  = float(value)
        mg = (hi - lo) * 0.2
        if   lo <= v <= hi          : return "normal",   "#16a34a"
        elif (lo-mg) <= v <= (hi+mg): return "warning",  "#d97706"
        else                         : return "critical", "#dc2626"
    except:
        return "info", "#64748b"


def fmt_val(v, fname):
    unit = NORMAL_RANGES[fname][2] if fname in NORMAL_RANGES else ""
    try:
        fv = float(v)
        s  = str(int(fv)) if fv == int(fv) else f"{fv:.2f}"
        return f"{s} {unit}".strip()
    except:
        return str(v)


def get_value_note(fname, value):
    if fname not in NORMAL_RANGES:
        return ""
    lo, hi, unit = NORMAL_RANGES[fname]
    try:
        v = float(value)
        if   v > hi : return f"↑ Above normal ({lo}–{hi} {unit})"
        elif v < lo : return f"↓ Below normal ({lo}–{hi} {unit})"
        else         : return f"✓ Within normal ({lo}–{hi} {unit})"
    except:
        return ""


# ══════════════════════════════════════════════════════════════
#   MAIN VISUALIZATION — Corporate Use-Case Flow
# ══════════════════════════════════════════════════════════════
def render_usecase_flow(key, feat_array, feat_names,
                         display_inputs, disease_label, pred, prob):

    risk_pct = prob[1] * 100
    n        = len(feat_names)
    sv       = safe_shap(key, feat_array, n)

    # Build ranked factor table
    abs_sv  = np.abs(sv)
    total   = abs_sv.sum() if abs_sv.sum() > 0 else 1
    pct_inv = abs_sv / total * 100

    rows = []
    for i, fname in enumerate(feat_names):
        raw_val = feat_array.flatten()[i]
        rows.append({
            "fname"     : fname,
            "label"     : LABELS.get(fname, fname),
            "raw_value" : raw_val,
            "shap"      : float(sv[i]),
            "abs_shap"  : float(abs_sv[i]),
            "pct"       : float(pct_inv[i]),
            "direction" : "risk" if sv[i] > 0 else "protective",
        })

    # Sort: positive SHAP (risk) first by magnitude,
    #       then negative SHAP (protective) by magnitude
    risk_rows = sorted([r for r in rows if r["shap"] >  0],
                       key=lambda x: x["abs_shap"], reverse=True)
    prot_rows = sorted([r for r in rows if r["shap"] <= 0],
                       key=lambda x: x["abs_shap"], reverse=True)
    sorted_rows = risk_rows + prot_rows

    # ── 1. INPUT CARD ─────────────────────────────────────────
    st.markdown('<div class="section-title">INPUT: Patient Data</div>',
                unsafe_allow_html=True)

    inp_cols = st.columns(4)
    for i, (k, v) in enumerate(display_inputs.items()):
        inp_cols[i % 4].markdown(
            f'<div style="background:white;border:1px solid #e2e8f0;'
            f'border-radius:10px;padding:12px 16px;margin:4px 0;'
            f'text-align:center;">'
            f'<div style="font-size:11px;color:#64748b;font-weight:500;'
            f'text-transform:uppercase;letter-spacing:0.5px">{k}</div>'
            f'<div style="font-size:20px;font-weight:700;color:#1e293b;'
            f'margin-top:4px">{v}</div>'
            f'</div>',
            unsafe_allow_html=True
        )

    # ── 2. FLOW ARROW ─────────────────────────────────────────
    st.markdown(
        '<div style="text-align:center;font-size:30px;'
        'color:#94a3b8;margin:10px 0">↓</div>',
        unsafe_allow_html=True)

    # ── 3. AI PROCESSING BOX ──────────────────────────────────
    st.markdown(
        '<div class="ai-box">'
        '<div style="font-size:32px;margin-bottom:8px">🤖</div>'
        '<div style="font-size:17px;font-weight:700;'
        'letter-spacing:0.5px">AI Model Processing</div>'
        '<div style="font-size:13px;opacity:0.8;margin-top:4px">'
        'Analyzing risk factors using Random Forest + SHAP Explainability'
        '</div></div>',
        unsafe_allow_html=True)

    # ── 4. FLOW ARROW ─────────────────────────────────────────
    st.markdown(
        '<div style="text-align:center;font-size:30px;'
        'color:#94a3b8;margin:10px 0">↓</div>',
        unsafe_allow_html=True)

    # ── 5. OUTPUT RISK RESULT ─────────────────────────────────
    if pred == 1:
        risk_css   = "result-high"
        risk_label = "HIGH RISK"
        risk_icon  = "⚠️"
        risk_color = "#dc2626"
    else:
        risk_css   = "result-low"
        risk_label = "LOW RISK"
        risk_icon  = "✅"
        risk_color = "#16a34a"

    st.markdown('<div class="section-title">OUTPUT: Risk Assessment</div>',
                unsafe_allow_html=True)

    col_res1, col_res2 = st.columns([2, 1])
    with col_res1:
        st.markdown(
            f'<div class="{risk_css}">'
            f'<div style="font-size:42px;font-weight:800;'
            f'color:{risk_color};line-height:1">{risk_pct:.0f}%</div>'
            f'<div style="font-size:14px;font-weight:600;'
            f'color:{risk_color};margin-top:4px">'
            f'{disease_label} Risk</div>'
            f'<div style="font-size:13px;color:#64748b;margin-top:8px">'
            f'{risk_icon} {risk_label} &nbsp;|&nbsp; '
            f'Confidence: {max(prob)*100:.1f}%</div>'
            f'</div>',
            unsafe_allow_html=True)

    with col_res2:
        # Mini donut-style gauge
        fig_g, ax_g = plt.subplots(figsize=(3, 3))
        fig_g.patch.set_facecolor("none")
        ax_g.set_facecolor("none")
        size  = 0.35
        outer = [risk_pct, 100 - risk_pct]
        c_out = [risk_color, "#e2e8f0"]
        ax_g.pie(outer, radius=1, colors=c_out, startangle=90,
                 wedgeprops=dict(width=size, edgecolor="white"))
        ax_g.text(0, 0, f"{risk_pct:.0f}%",
                  ha="center", va="center",
                  fontsize=18, fontweight="bold",
                  color=risk_color)
        ax_g.text(0, -0.28, "Risk Score",
                  ha="center", va="center",
                  fontsize=8, color="#64748b")
        st.pyplot(fig_g, use_container_width=False)
        plt.close()

    st.markdown("---")

    # ── 6. RANKED FACTOR TABLE ────────────────────────────────
    st.markdown('<div class="section-title">Factor Analysis — Ranked by Impact</div>',
                unsafe_allow_html=True)

    st.markdown(
        '<div style="background:#f8fafc;border:1px solid #e2e8f0;'
        'border-radius:10px;padding:10px 14px;margin-bottom:10px;'
        'font-size:12px;color:#64748b">'
        '🔴 <b style="color:#dc2626">Red bar</b> = Factor INCREASES disease risk &nbsp;|&nbsp; '
        '🔵 <b style="color:#2563a8">Blue bar</b> = Factor DECREASES / protects against risk &nbsp;|&nbsp; '
        'Ranked from highest to lowest impact'
        '</div>',
        unsafe_allow_html=True)

    max_pct = max([r["pct"] for r in sorted_rows]) if sorted_rows else 1

    for rank, row in enumerate(sorted_rows, 1):
        fname   = row["fname"]
        label   = row["label"]
        pct     = row["pct"]
        raw     = row["raw_value"]
        shap_v  = row["shap"]
        is_risk = row["direction"] == "risk"

        bar_color  = "#dc2626" if is_risk else "#2563a8"
        bg_color   = "#fff5f5" if is_risk else "#eff6ff"
        border_col = "#dc2626" if is_risk else "#2563a8"
        direction_label = "↑ Increases Risk" if is_risk else "↓ Decreases Risk"
        direction_color = "#dc2626" if is_risk else "#2563a8"

        # Status pill
        status, stat_color = get_status(fname, raw)
        pill_css = {"normal":"pill-normal","warning":"pill-warning",
                    "critical":"pill-critical","info":"pill-normal"}.get(status,"pill-normal")
        status_label = {"normal":"Normal","warning":"Warning",
                        "critical":"Critical","info":"—"}.get(status,"—")

        val_str  = fmt_val(raw, fname)
        val_note = get_value_note(fname, raw)
        bar_w    = (pct / max_pct) * 100

        st.markdown(f"""
        <div style="background:{bg_color};border-left:5px solid {border_col};
             border-radius:0 10px 10px 0;padding:12px 16px;
             margin:5px 0;box-shadow:0 1px 3px rgba(0,0,0,0.06)">

          <div style="display:flex;justify-content:space-between;
                      align-items:center;margin-bottom:6px">
            <div>
              <span style="font-size:13px;font-weight:700;color:#1e293b">
                #{rank} &nbsp; {label}
              </span>
              <span style="font-size:11px;font-weight:600;
                    background:{'#fee2e2' if is_risk else '#dbeafe'};
                    color:{direction_color};padding:2px 8px;
                    border-radius:20px;margin-left:8px">
                {direction_label}
              </span>
              <span style="font-size:11px;
                    background:{'#fee2e2' if status=='critical'
                                else '#fef9c3' if status=='warning'
                                else '#dcfce7'};
                    color:{stat_color};padding:2px 8px;
                    border-radius:20px;margin-left:6px;font-weight:600">
                {status_label}
              </span>
            </div>
            <div style="text-align:right">
              <span style="font-size:18px;font-weight:800;
                    color:{direction_color}">{pct:.1f}%</span>
              <span style="font-size:10px;color:#94a3b8;
                    display:block">involvement</span>
            </div>
          </div>

          <!-- Progress bar -->
          <div style="background:#e2e8f0;border-radius:4px;
                      height:8px;margin-bottom:8px">
            <div style="background:{bar_color};border-radius:4px;
                        height:8px;width:{bar_w:.1f}%"></div>
          </div>

          <!-- Value row -->
          <div style="display:flex;gap:24px;font-size:12px;color:#475569">
            <span>📊 <b>Actual Value:</b>
              <span style="color:{stat_color};font-weight:700">
                {val_str}
              </span>
            </span>
            <span style="color:#64748b">{val_note}</span>
            <span style="color:#94a3b8">
              SHAP: {shap_v:+.4f}
            </span>
          </div>

        </div>
        """, unsafe_allow_html=True)

    st.markdown("---")

    # ── 7. VISUALIZATION CHARTS ───────────────────────────────
    st.markdown('<div class="section-title">Explainability Charts</div>',
                unsafe_allow_html=True)

    risk_rows_chart = [r for r in sorted_rows if r["shap"] > 0][:8]
    prot_rows_chart = [r for r in sorted_rows if r["shap"] <= 0][:5]
    chart_rows      = risk_rows_chart + prot_rows_chart

    fig, axes = plt.subplots(1, 3, figsize=(18, max(6, len(chart_rows)*0.55+2)))
    fig.patch.set_facecolor("#f8fafc")
    for ax in axes:
        ax.set_facecolor("white")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_color("#e2e8f0")
        ax.spines["bottom"].set_color("#e2e8f0")

    # Chart 1: % Involvement ranked bar
    labels_c  = [r["label"] for r in chart_rows]
    pcts_c    = [r["pct"] for r in chart_rows]
    colors_c  = ["#dc2626" if r["shap"]>0 else "#2563a8"
                 for r in chart_rows]

    plot_order = list(range(len(chart_rows)))[::-1]
    labels_p   = [labels_c[i] for i in plot_order]
    pcts_p     = [pcts_c[i]   for i in plot_order]
    colors_p   = [colors_c[i] for i in plot_order]

    bars1 = axes[0].barh(labels_p, pcts_p, color=colors_p,
                          edgecolor="white", height=0.65)
    axes[0].set_xlabel("% Involvement", fontsize=10, color="#475569")
    axes[0].set_title("Factor Involvement (%)",
                      fontsize=12, fontweight="bold", color="#1e293b", pad=12)
    axes[0].tick_params(axis="y", labelsize=9, colors="#475569")
    axes[0].tick_params(axis="x", labelsize=9, colors="#475569")
    for bar, pct_v in zip(bars1, pcts_p):
        axes[0].text(bar.get_width() + 0.3,
                     bar.get_y() + bar.get_height()/2,
                     f"{pct_v:.1f}%",
                     va="center", fontsize=8.5,
                     fontweight="bold", color="#1e293b")
    red_p  = mpatches.Patch(color="#dc2626", label="Increases Risk")
    blue_p = mpatches.Patch(color="#2563a8", label="Decreases Risk")
    axes[0].legend(handles=[red_p, blue_p],
                   fontsize=8, framealpha=0.9,
                   loc="lower right")
    axes[0].set_xlim(0, max(pcts_p)*1.25 if pcts_p else 10)

    # Chart 2: SHAP Value Direction
    shap_vals_c = [r["shap"] for r in chart_rows]
    sv_order    = list(range(len(chart_rows)))[::-1]
    labels_sv   = [labels_c[i] for i in sv_order]
    shaps_sv    = [shap_vals_c[i] for i in sv_order]
    colors_sv   = ["#dc2626" if x > 0 else "#2563a8" for x in shaps_sv]

    bars2 = axes[1].barh(labels_sv, shaps_sv,
                          color=colors_sv, edgecolor="white", height=0.65)
    axes[1].axvline(x=0, color="#94a3b8", linewidth=1.2, linestyle="-")
    axes[1].set_xlabel("SHAP Value (Impact Direction)",
                       fontsize=10, color="#475569")
    axes[1].set_title("Risk Direction\n(Which way each factor pushes)",
                      fontsize=12, fontweight="bold", color="#1e293b", pad=12)
    axes[1].tick_params(axis="y", labelsize=9, colors="#475569")
    axes[1].tick_params(axis="x", labelsize=9, colors="#475569")
    for bar, sv_v in zip(bars2, shaps_sv):
        ha   = "left"  if sv_v >= 0 else "right"
        xpos = bar.get_width() + 0.001 if sv_v >= 0 else bar.get_width() - 0.001
        axes[1].text(xpos, bar.get_y() + bar.get_height()/2,
                     f" {sv_v:+.3f}",
                     va="center", ha=ha, fontsize=8,
                     color="#1e293b")

    # Chart 3: Top risk factors pie
    top_risk_chart = [r for r in sorted_rows if r["shap"] > 0][:6]
    if top_risk_chart:
        pie_labels = [r["label"] for r in top_risk_chart]
        pie_vals   = [r["pct"]   for r in top_risk_chart]
        others_pct = 100 - sum(pie_vals)
        if others_pct > 0.5:
            pie_labels.append("Other Factors")
            pie_vals.append(others_pct)

        RED_PALETTE = ["#dc2626","#ef4444","#f87171",
                       "#fca5a5","#fecaca","#fee2e2","#e2e8f0"]

        wedges, texts, autotexts = axes[2].pie(
            pie_vals,
            labels=pie_labels,
            autopct=lambda p: f"{p:.1f}%" if p > 4 else "",
            colors=RED_PALETTE[:len(pie_vals)],
            startangle=90,
            pctdistance=0.72,
            wedgeprops=dict(edgecolor="white", linewidth=2)
        )
        for t in autotexts:
            t.set_fontsize(8.5)
            t.set_fontweight("bold")
            t.set_color("white")
        for t in texts:
            t.set_fontsize(8.5)
            t.set_color("#475569")
        axes[2].set_title("Risk Factor Distribution\n(Disease-Causing Factors Only)",
                          fontsize=12, fontweight="bold",
                          color="#1e293b", pad=12)

    plt.suptitle(f"Complete Explainability Report — {disease_label}",
                 fontsize=14, fontweight="bold",
                 color="#1e3a5f", y=1.01)
    plt.tight_layout()
    st.pyplot(fig)
    plt.close()

    # ── 8. PLAIN ENGLISH SUMMARY ──────────────────────────────
    st.markdown('<div class="section-title">Plain English Explanation</div>',
                unsafe_allow_html=True)
    st.markdown('<div class="summary-box">', unsafe_allow_html=True)

    for i, row in enumerate(sorted_rows[:5], 1):
        is_risk   = row["shap"] > 0
        color     = "#dc2626" if is_risk else "#2563a8"
        arrow     = "↑" if is_risk else "↓"
        direction = "increases" if is_risk else "decreases"
        val_note  = get_value_note(row["fname"], row["raw_value"])
        val_str   = fmt_val(row["raw_value"], row["fname"])

        st.markdown(
            f'<div style="padding:8px 0;border-bottom:1px solid #f1f5f9">'
            f'<span style="font-weight:700;color:{color}">'
            f'{arrow} #{i}. {row["label"]}</span> — '
            f'<span style="color:#1e293b">Value: <b>{val_str}</b>. '
            f'{val_note}.</span> '
            f'<span style="color:{color};font-weight:600">'
            f'This {direction} the risk by {row["pct"]:.1f}%.</span>'
            f'</div>',
            unsafe_allow_html=True
        )

    st.markdown('</div>', unsafe_allow_html=True)
    st.caption(
        "⚠️ Educational use only. SHAP values explain model decisions. "
        "Always consult a qualified medical professional."
    )


# ══════════════════════════════════════════════════════════════
#   HEADER
# ══════════════════════════════════════════════════════════════
st.markdown(
    '<div style="text-align:center;padding:10px 0 4px 0">'
    '<h1 style="color:#1e3a5f;font-weight:800;font-size:28px;margin:0">'
    '🏥 Multiple Disease Prediction System</h1>'
    '<p style="color:#64748b;font-size:14px;margin:6px 0 0 0">'
    'AI-Powered &nbsp;·&nbsp; Explainable AI (SHAP) &nbsp;·&nbsp; '
    'Clinical-Grade Parameter Analysis &nbsp;·&nbsp; Educational Use Only'
    '</p></div>',
    unsafe_allow_html=True
)
st.markdown("---")

mode = st.radio(
    "Prediction Mode:",
    ["🔬 Single Disease — Full Use-Case Flow",
     "🧬 All 4 Diseases at Once"],
    horizontal=True
)
st.markdown("---")


# ══════════════════════════════════════════════════════════════
#   MODE 1: SINGLE DISEASE
# ══════════════════════════════════════════════════════════════
if mode == "🔬 Single Disease — Full Use-Case Flow":

    disease = st.selectbox("Select Disease to Predict:",
        ["Diabetes","Heart Disease","Kidney Disease","Liver Disease"])
    st.markdown("---")

    # ── DIABETES ─────────────────────────────────────────────
    if disease == "Diabetes":
        st.markdown("### 🩸 Diabetes Risk Assessment")
        c1, c2 = st.columns(2)
        with c1:
            gender  = st.selectbox("Gender",["Female","Male","Other"])
            age     = st.number_input("Age (years)", 1, 120, 45)
            hypert  = st.selectbox("Hypertension",["No","Yes"])
            hd      = st.selectbox("Heart Disease",["No","Yes"])
        with c2:
            smoking = st.selectbox("Smoking History",
                ["never","No Info","current","former","ever","not current"])
            bmi     = st.number_input("BMI (kg/m²)", 10.0,70.0,25.0,step=0.1)
            hba1c   = st.number_input("HbA1c Level (%)",3.0,15.0,5.5,step=0.1)
            glucose = st.number_input("Blood Glucose (mg/dL)",50,400,100)

        if st.button("🔮 Predict & Explain Diabetes"):
            gm = {"Female":0,"Male":1,"Other":2}
            sm = {"No Info":0,"current":1,"ever":2,
                  "former":3,"never":4,"not current":5}
            feat = np.array([[gm[gender],age,
                              1 if hypert=="Yes" else 0,
                              1 if hd=="Yes" else 0,
                              sm[smoking],bmi,hba1c,glucose]])
            pred, prob = safe_predict("diabetes", feat)
            st.markdown("---")
            render_usecase_flow(
                "diabetes", feat, FEATURES["diabetes"],
                {"Age":age, "BMI":f"{bmi}", "HbA1c":f"{hba1c}%",
                 "Blood Glucose":f"{glucose} mg/dL",
                 "Hypertension":hypert, "Smoking":smoking},
                "Diabetes", pred, prob
            )

    # ── HEART ────────────────────────────────────────────────
    elif disease == "Heart Disease":
        st.markdown("### ❤️ Heart Disease Risk Assessment")
        c1, c2 = st.columns(2)
        with c1:
            age     = st.number_input("Age (years)",1,120,55)
            sex     = st.selectbox("Sex",["Female","Male"])
            cp      = st.selectbox("Chest Pain Type",
                        ["Typical angina","Atypical angina",
                         "Non-anginal pain","Asymptomatic"])
            rbp     = st.number_input("Resting BP (mmHg)",80,220,130)
            chol    = st.number_input("Cholesterol (mg/dL)",100,600,200)
            fbs     = st.selectbox("Fasting Blood Sugar",
                        ["Lower than 120 mg/ml","Greater than 120 mg/ml"])
            recg    = st.selectbox("Resting ECG",
                        ["Normal","ST-T wave abnormality",
                         "Left ventricular hypertrophy"])
        with c2:
            mhr     = st.number_input("Max Heart Rate (bpm)",50,250,150)
            exang   = st.selectbox("Exercise Induced Angina",["No","Yes"])
            oldpeak = st.number_input("ST Depression",0.0,7.0,1.0,step=0.1)
            slope   = st.selectbox("ST Slope",
                        ["Upsloping","Flat","Downsloping"])
            vessels = st.selectbox("Vessels Colored",
                        ["Zero","One","Two","Three"])
            thal    = st.selectbox("Thalassemia",
                        ["Normal","Fixed Defect","Reversable Defect"])

        if st.button("🔮 Predict & Explain Heart Disease"):
            cp_m={"Asymptomatic":0,"Atypical angina":1,
                  "Non-anginal pain":2,"Typical angina":3}
            fb_m={"Greater than 120 mg/ml":0,"Lower than 120 mg/ml":1}
            re_m={"Left ventricular hypertrophy":0,"Normal":1,
                  "ST-T wave abnormality":2}
            sl_m={"Downsloping":0,"Flat":1,"Upsloping":2}
            ve_m={"One":0,"Three":1,"Two":2,"Zero":3}
            th_m={"Fixed Defect":0,"Normal":1,"Reversable Defect":2}
            feat = np.array([[age,0 if sex=="Female" else 1,
                              cp_m[cp],rbp,chol,fb_m[fbs],re_m[recg],
                              mhr,0 if exang=="No" else 1,oldpeak,
                              sl_m[slope],ve_m[vessels],th_m[thal]]])
            pred, prob = safe_predict("heart", feat)
            st.markdown("---")
            render_usecase_flow(
                "heart", feat, FEATURES["heart"],
                {"Age":age,"Sex":sex,"BP":f"{rbp} mmHg",
                 "Cholesterol":f"{chol} mg/dL",
                 "Max HR":f"{mhr} bpm","ST Depression":oldpeak},
                "Heart Disease", pred, prob
            )

    # ── KIDNEY ───────────────────────────────────────────────
    elif disease == "Kidney Disease":
        st.markdown("### 🫘 Kidney Disease Risk Assessment")
        c1, c2 = st.columns(2)
        with c1:
            age  = st.number_input("Age",1,120,48)
            bp   = st.number_input("Blood Pressure",50,180,80)
            sg   = st.selectbox("Specific Gravity",
                       [1.005,1.010,1.015,1.020,1.025])
            al   = st.selectbox("Albumin (0-5)",[0,1,2,3,4,5])
            su   = st.selectbox("Sugar (0-5)",[0,1,2,3,4,5])
            rbc  = st.selectbox("Red Blood Cells",["Normal","Abnormal"])
            pc   = st.selectbox("Pus Cells",["Normal","Abnormal"])
            pcc  = st.selectbox("Pus Cell Clumps",["Not Present","Present"])
            ba   = st.selectbox("Bacteria",["Not Present","Present"])
            bgr  = st.number_input("Blood Glucose (mg/dL)",50,500,120)
            bu   = st.number_input("Blood Urea (mg/dL)",5,200,40)
            sc   = st.number_input("Serum Creatinine",0.1,20.0,1.2,step=0.1)
        with c2:
            sod  = st.number_input("Sodium",100,170,138)
            pot  = st.number_input("Potassium",2.0,10.0,4.5,step=0.1)
            hemo = st.number_input("Hemoglobin",3.0,20.0,15.0,step=0.1)
            pcv  = st.number_input("Packed Cell Volume",10,60,44)
            wc   = st.number_input("White Blood Cells",2000,20000,7800)
            rc   = st.number_input("Red Blood Cell Count",1.0,8.0,5.2,step=0.1)
            htn  = st.selectbox("Hypertension",["No","Yes"])
            dm   = st.selectbox("Diabetes Mellitus",["No","Yes"])
            cad  = st.selectbox("Coronary Artery Disease",["No","Yes"])
            appet= st.selectbox("Appetite",["Good","Poor"])
            pe   = st.selectbox("Pedal Edema",["No","Yes"])
            ane  = st.selectbox("Anemia",["No","Yes"])

        if st.button("🔮 Predict & Explain Kidney Disease"):
            yn={"No":0,"Yes":1}
            feat=np.array([[age,bp,sg,al,su,
                            1 if rbc=="Normal" else 0,
                            1 if pc=="Normal" else 0,
                            1 if pcc=="Present" else 0,
                            1 if ba=="Present" else 0,
                            bgr,bu,sc,sod,pot,hemo,
                            pcv,wc,rc,yn[htn],yn[dm],yn[cad],
                            0 if appet=="Good" else 1,yn[pe],yn[ane]]])
            pred, prob = safe_predict("kidney", feat)
            st.markdown("---")
            render_usecase_flow(
                "kidney", feat, FEATURES["kidney"],
                {"Age":age,"BP":f"{bp} mmHg",
                 "Hemoglobin":f"{hemo} g/dL",
                 "Creatinine":f"{sc} mg/dL",
                 "Blood Urea":f"{bu} mg/dL",
                 "Blood Glucose":f"{bgr} mg/dL"},
                "Kidney Disease", pred, prob
            )

    # ── LIVER ────────────────────────────────────────────────
    elif disease == "Liver Disease":
        st.markdown("### 🫀 Liver Disease Risk Assessment")
        c1, c2 = st.columns(2)
        with c1:
            age     = st.number_input("Age",1,120,45)
            gender  = st.selectbox("Gender",["Female","Male"])
            tot_bil = st.number_input("Total Bilirubin (mg/dL)",
                                      0.1,80.0,1.0,step=0.1)
            dir_bil = st.number_input("Direct Bilirubin (mg/dL)",
                                      0.1,40.0,0.3,step=0.1)
            tot_pro = st.number_input("Total Proteins",50,800,200)
        with c2:
            alb     = st.number_input("Albumin",5,100,35)
            ag_r    = st.number_input("A/G Ratio",0.1,3.0,1.0,step=0.1)
            sgpt    = st.number_input("SGPT (ALT) U/L",5.0,2000.0,40.0)
            sgot    = st.number_input("SGOT (AST) U/L",5.0,3000.0,35.0)
            alkp    = st.number_input("Alkaline Phosphatase U/L",50,2200,200)

        if st.button("🔮 Predict & Explain Liver Disease"):
            feat=np.array([[age,0 if gender=="Female" else 1,
                            tot_bil,dir_bil,tot_pro,
                            alb,ag_r,sgpt,sgot,alkp]])
            pred, prob = safe_predict("liver", feat)
            st.markdown("---")
            render_usecase_flow(
                "liver", feat, FEATURES["liver"],
                {"Age":age,"Gender":gender,
                 "Total Bilirubin":f"{tot_bil} mg/dL",
                 "SGPT":f"{sgpt} U/L",
                 "SGOT":f"{sgot} U/L",
                 "Alk Phosphatase":f"{alkp} U/L"},
                "Liver Disease", pred, prob
            )


# ══════════════════════════════════════════════════════════════
#   MODE 2: ALL 4 DISEASES
# ══════════════════════════════════════════════════════════════
else:
    st.markdown("### 🧬 All 4 Diseases — Simultaneous Assessment")
    st.info("Enter your health parameters. All 4 disease risks will be assessed and explained simultaneously.")

    c1,c2,c3 = st.columns(3)
    with c1:
        age    = st.number_input("Age (years)",1,120,45)
        gender = st.selectbox("Gender",["Female","Male"])
    with c2:
        bmi    = st.number_input("BMI",10.0,70.0,25.0,step=0.1)
        hypert = st.selectbox("Hypertension",["No","Yes"])
    with c3:
        hd     = st.selectbox("Heart Disease History",["No","Yes"])
        smoking= st.selectbox("Smoking",
                    ["never","No Info","current","former","ever","not current"])

    c1,c2,c3,c4 = st.columns(4)
    with c1:
        glucose= st.number_input("Blood Glucose (mg/dL)",50,500,120)
        hba1c  = st.number_input("HbA1c (%)",3.0,15.0,5.5,step=0.1)
        bu     = st.number_input("Blood Urea (mg/dL)",5,200,40)
    with c2:
        sc     = st.number_input("Serum Creatinine",0.1,20.0,1.2,step=0.1)
        hemo   = st.number_input("Hemoglobin (g/dL)",3.0,20.0,15.0,step=0.1)
        chol   = st.number_input("Cholesterol (mg/dL)",100,600,200)
    with c3:
        tot_bil= st.number_input("Total Bilirubin",0.1,80.0,1.0,step=0.1)
        sgpt   = st.number_input("SGPT (U/L)",5.0,2000.0,40.0)
        sgot   = st.number_input("SGOT (U/L)",5.0,3000.0,35.0)
    with c4:
        bp     = st.number_input("Blood Pressure (mmHg)",50,220,80)
        mhr    = st.number_input("Max Heart Rate",50,250,150)
        alkp   = st.number_input("Alkaline Phosphatase",50,2200,200)

    st.markdown("---")

    if st.button("🚀 Assess All 4 Diseases"):
        gm={"Female":0,"Male":1}
        sm={"No Info":0,"current":1,"ever":2,
            "former":3,"never":4,"not current":5}
        yn={"No":0,"Yes":1}

        feat_d=np.array([[gm[gender],age,yn[hypert],yn[hd],
                          sm[smoking],bmi,hba1c,glucose]])
        feat_h=np.array([[age,gm[gender],3,bp,chol,1,1,
                          mhr,0,1.0,2,3,1]])
        feat_k=np.array([[age,bp,1.020,0,0,1,1,0,0,
                          glucose,bu,sc,138,4.5,hemo,
                          44,7800,5.2,yn[hypert],yn[hd],0,0,0,0]])
        feat_l=np.array([[age,gm[gender],tot_bil,tot_bil*0.3,
                          220,35,1.1,sgpt,sgot,alkp]])

        pd_,pb_d = safe_predict("diabetes",feat_d)
        ph_,pb_h = safe_predict("heart",   feat_h)
        pk_,pb_k = safe_predict("kidney",  feat_k)
        pl_,pb_l = safe_predict("liver",   feat_l)

        scores = {
            "🩸 Diabetes"      : pb_d[1]*100,
            "❤️ Heart Disease" : pb_h[1]*100,
            "🫘 Kidney Disease": pb_k[1]*100,
            "🫀 Liver Disease" : pb_l[1]*100,
        }
        preds = [pd_, ph_, pk_, pl_]

        # Summary cards
        st.markdown("### 📊 Risk Summary — All 4 Diseases")
        cols4 = st.columns(4)
        for col,(name,score),pred in zip(cols4,scores.items(),preds):
            color = "#dc2626" if pred==1 else "#16a34a"
            bg    = "#fff5f5" if pred==1 else "#f0fdf4"
            icon  = "⚠️"      if pred==1 else "✅"
            col.markdown(
                f'<div style="border:2px solid {color};'
                f'border-radius:12px;padding:16px;'
                f'text-align:center;background:{bg}">'
                f'<div style="font-size:26px">{icon}</div>'
                f'<div style="font-weight:700;font-size:12px;'
                f'color:{color};margin:4px 0">{name}</div>'
                f'<div style="font-size:26px;color:{color};'
                f'font-weight:800">{score:.1f}%</div>'
                f'<div style="font-size:11px;color:#94a3b8">'
                f'risk score</div></div>',
                unsafe_allow_html=True)

        st.markdown("---")

        # Overview chart
        st.markdown("### 📈 Risk Overview")
        fig_ov, ax_ov = plt.subplots(figsize=(12,4))
        fig_ov.patch.set_facecolor("#f8fafc")
        ax_ov.set_facecolor("white")
        ax_ov.spines["top"].set_visible(False)
        ax_ov.spines["right"].set_visible(False)

        dnames = list(scores.keys())
        dscores= list(scores.values())
        bcolors= ["#dc2626" if s>=50 else "#f97316" if s>=30
                  else "#16a34a" for s in dscores]

        bars_ov = ax_ov.barh(dnames, dscores,
                              color=bcolors, edgecolor="white", height=0.5)
        ax_ov.axvline(x=50, color="#dc2626", linewidth=1.5,
                      linestyle="--", alpha=0.6, label="Risk Threshold (50%)")
        ax_ov.set_xlim(0,110)
        ax_ov.set_xlabel("Risk Score (%)", fontsize=11, color="#475569")
        ax_ov.set_title("Disease Risk Scores Overview",
                         fontsize=13, fontweight="bold", color="#1e3a5f")
        ax_ov.legend(fontsize=9)
        for b,s in zip(bars_ov,dscores):
            ax_ov.text(b.get_width()+1,
                       b.get_y()+b.get_height()/2,
                       f"{s:.1f}%", va="center",
                       fontsize=12, fontweight="bold",
                       color="#1e293b")
        plt.tight_layout()
        st.pyplot(fig_ov)
        plt.close()

        st.markdown("---")
        st.markdown("### 🧠 Detailed Explanation — All 4 Diseases")

        tab1,tab2,tab3,tab4 = st.tabs(
            ["🩸 Diabetes","❤️ Heart Disease",
             "🫘 Kidney Disease","🫀 Liver Disease"])

        with tab1:
            render_usecase_flow(
                "diabetes",feat_d,FEATURES["diabetes"],
                {"Age":age,"BMI":bmi,"HbA1c":f"{hba1c}%",
                 "Blood Glucose":f"{glucose} mg/dL"},
                "Diabetes",pd_,pb_d)
        with tab2:
            render_usecase_flow(
                "heart",feat_h,FEATURES["heart"],
                {"Age":age,"BP":f"{bp} mmHg",
                 "Cholesterol":f"{chol} mg/dL",
                 "Max HR":f"{mhr} bpm"},
                "Heart Disease",ph_,pb_h)
        with tab3:
            render_usecase_flow(
                "kidney",feat_k,FEATURES["kidney"],
                {"Age":age,"Hemoglobin":f"{hemo} g/dL",
                 "Creatinine":f"{sc} mg/dL",
                 "Blood Urea":f"{bu} mg/dL"},
                "Kidney Disease",pk_,pb_k)
        with tab4:
            render_usecase_flow(
                "liver",feat_l,FEATURES["liver"],
                {"Age":age,"Total Bilirubin":f"{tot_bil} mg/dL",
                 "SGPT":f"{sgpt} U/L","SGOT":f"{sgot} U/L"},
                "Liver Disease",pl_,pb_l)

    st.markdown("---")
    st.markdown(
        '<div style="text-align:center;color:#94a3b8;font-size:12px">'
        '⚠️ For educational purposes only. '
        'Always consult a qualified doctor for diagnosis.</div>',
        unsafe_allow_html=True)
'''

with open("app.py","w") as f:
    f.write(app_code)

print("✅ Corporate-grade app.py written!")
print("\n🆕 Matches your reference image:")
print("  📦 INPUT card → AI Processing → OUTPUT flow")
print("  🔴 Red = risk factors | 🔵 Blue = protective factors")
print("  📊 Ranked list: #1 most impactful → last least impactful")
print("  📏 Progress bar per factor showing % involvement")
print("  🏷️  Actual value + normal range + status pill per factor")
print("  🥧 Pie chart shows only disease-causing factors (red shades)")
print("  🔍 Plain English: 'Your cholesterol (240 mg/dL) is above normal'")

In [ ]:
# CELL 73: Launch upgraded app

from pyngrok import ngrok
import subprocess, time

NGROK_TOKEN = "35CXEdaL4DIgiHpOybH6DbgYKWp_3SxMQF3d9FFRcU33DnYZc"   # ← paste your token here

ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()

process = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.headless", "true",
     "--server.enableCORS", "false",
     "--server.enableXsrfProtection", "false"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(6)
public_url = ngrok.connect(8501)

print("=" * 55)
print("   🚀  UPGRADED APP IS LIVE!")
print("=" * 55)
print(f"\n🌐  Open this link:\n\n    👉  {public_url}\n")
print("✨  New Features Available:")
print("    🔬 Single Disease + SHAP explanation")
print("    🧬 All 4 diseases predicted at once")
print("    📊 Risk bar chart + pie chart")
print("    🧠 SHAP tabs for all 4 diseases")
print("=" * 55)
